In [1]:
import os
import re
import gc
import json
import numpy as np
import pandas as pd

from typing import List, Tuple, Dict, Optional
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, f1_score, precision_score, recall_score
from catboost import CatBoostClassifier, Pool

# =========================
# 配置
# =========================
DATA_PATH = "train/train.csv"  # ← 修改为你的训练集路径
TEST_DATA_PATH = "testaa/testaa.csv"
TARGET_CANDIDATES = ["label", "target", "is_default", "default", "y", "bad", "risk_flag"]
RANDOM_SEED = 42
N_FOLDS = 5
USE_GPU = True  # 若有GPU且安装对应版本CatBoost，可设为True

# =========================
# 工具函数
# =========================
def infer_target_col(df: pd.DataFrame,
                     candidates: List[str]) -> str:
    for c in candidates:
        if c in df.columns:
            return c
    # 兜底：二分类列中优先包含 default/label/target 词根者
    binary_like = []
    for c in df.columns:
        if str(c).lower() in ["id", "uuid", "index", "loan_id", "user_id"]:
            continue
        vc = df[c].dropna().unique()
        if len(vc) == 2:
            binary_like.append(c)
    preferred = sorted(binary_like, key=lambda x: (0 if any(k in str(x).lower() for k in ["default","label","target"]) else 1, str(x)))
    return preferred[0] if preferred else None

def detect_datetime_cols(df: pd.DataFrame,
                         sample_size: int = 5000,
                         threshold: float = 0.8) -> List[str]:
    dt_cols = []
    for c in df.columns:
        s = df[c]
        if s.dtype == "object" or np.issubdtype(s.dtype, np.integer) or np.issubdtype(s.dtype, np.floating):
            sample = s.dropna().astype(str).head(sample_size)
            if sample.empty: 
                continue
            parsed = pd.to_datetime(sample, errors="coerce", infer_datetime_format=True)
            if parsed.notna().mean() >= threshold:
                dt_cols.append(c)
    return dt_cols

def get_low_card_int_as_cat(df: pd.DataFrame, target: Optional[str]) -> List[str]:
    res = []
    for c in df.columns:
        if c == target: 
            continue
        if np.issubdtype(df[c].dtype, np.number):
            uniq = df[c].nunique(dropna=True)
            if 0 < uniq <= max(100, int(0.03 * len(df))):
                res.append(c)
    return res

def add_time_features(df: pd.DataFrame, dt_cols: List[str]) -> Tuple[pd.DataFrame, List[str]]:
    added = []
    for c in dt_cols:
        s = pd.to_datetime(df[c], errors="coerce", infer_datetime_format=True)
        df[f"{c}_year"] = s.dt.year
        df[f"{c}_month"] = s.dt.month
        df[f"{c}_day"] = s.dt.day
        df[f"{c}_dow"] = s.dt.dayofweek
        df[f"{c}_is_month_start"] = s.dt.is_month_start.astype("Int8")
        df[f"{c}_is_month_end"] = s.dt.is_month_end.astype("Int8")
        df[f"{c}_quarter"] = s.dt.quarter
        df[f"{c}_hour"] = s.dt.hour
        # days since min timestamp
        if s.notna().any():
            min_ts = s.min()
            df[f"{c}_days_since_min"] = (s - min_ts).dt.days
        added += [f"{c}_year", f"{c}_month", f"{c}_day", f"{c}_dow",
                  f"{c}_is_month_start", f"{c}_is_month_end",
                  f"{c}_quarter", f"{c}_hour", f"{c}_days_since_min"]
        # 原始时间列通常不直接用于CatBoost（可删除或保留备查）
        df.drop(columns=[c], inplace=True)
    return df, added

def add_missing_indicators(df: pd.DataFrame, thr_low=0.03, thr_high=0.95) -> List[str]:
    added = []
    miss_rate = df.isna().mean()
    for c, r in miss_rate.items():
        if thr_low <= r <= thr_high:
            ind = f"{c}_isna"
            df[ind] = df[c].isna().astype("Int8")
            added.append(ind)
    return added

def pick_first_col(cols: List[str], patterns: List[str]) -> Optional[str]:
    # 返回第一个“列名包含任一关键词”的列
    for p in patterns:
        for c in cols:
            if p in c.lower():
                return c
    return None

def safe_ratio(u: pd.Series, v: pd.Series) -> pd.Series:
    # 安全比率：避免除零/极端放大
    out = u.astype(float) / np.where(v.astype(float) == 0, np.nan, v.astype(float))
    # 截断极端值
    return out.clip(lower=-1e6, upper=1e6)

def build_ratio_features(df: pd.DataFrame) -> List[str]:
    """
    基于列名关键词自动构造有限且高价值的比率特征。
    """
    added = []
    cols = df.columns.tolist()
    lower_cols = [c.lower() for c in cols]

    def has_any(patterns: List[str]) -> bool:
        return any(any(p in c for p in patterns) for c in lower_cols)

    # 按语义组找列
    income_col   = pick_first_col(cols, ["income", "salary"])
    debt_col     = pick_first_col(cols, ["debt", "liab"])
    balance_col  = pick_first_col(cols, ["balance", "bal"])
    limit_col    = pick_first_col(cols, ["limit", "credit_limit"])
    amount_col   = pick_first_col(cols, ["amount", "amt", "principal", "loan_am", "funded"])
    payment_col  = pick_first_col(cols, ["payment", "installment", "repay"])
    bill_col     = pick_first_col(cols, ["bill"])

    # 已存在的“余额/授信比”
    existing_balance_limit = pick_first_col(cols, ["balance_limit"])

    # debt / income
    if debt_col and income_col:
        name = "debt_to_income"
        df[name] = safe_ratio(df[debt_col], df[income_col])
        added.append(name)

    # balance / limit
    if balance_col and limit_col and (existing_balance_limit is None):
        name = "balance_to_limit"
        df[name] = safe_ratio(df[balance_col], df[limit_col])
        added.append(name)

    # amount / income
    if amount_col and income_col:
        name = "amount_to_income"
        df[name] = safe_ratio(df[amount_col], df[income_col])
        added.append(name)

    # payment / income
    if payment_col and income_col:
        name = "payment_to_income"
        df[name] = safe_ratio(df[payment_col], df[income_col])
        added.append(name)

    # payment / balance
    if payment_col and balance_col:
        name = "payment_to_balance"
        df[name] = safe_ratio(df[payment_col], df[balance_col])
        added.append(name)

    # bill / income
    if bill_col and income_col:
        name = "bill_to_income"
        df[name] = safe_ratio(df[bill_col], df[income_col])
        added.append(name)

    return added

def cast_special_categoricals(df: pd.DataFrame) -> List[str]:
    """
    某些“数值形态但本质为类别”的特殊列统一转字符串（例如邮编等），避免数值顺序误导模型。
    """
    cat_special = []
    for c in df.columns:
        name = c.lower()
        if any(k in name for k in ["zip", "zipcode", "postal"]):
            df[c] = df[c].astype("Int64").astype(str)
            cat_special.append(c)
    return cat_special

def ks_score(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    return float(np.max(np.abs(tpr - fpr)))



In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
df_test = pd.read_csv(TEST_DATA_PATH, low_memory=False)
print(f"[INFO] Data shape: {df.shape}")

# 1) 目标列
target = infer_target_col(df, TARGET_CANDIDATES)
assert target is not None, "未能自动识别目标列，请在 TARGET_CANDIDATES 中补充列名。"
print(f"[INFO] Target column: {target}")

# 2) 时间列检测 & 衍生
dt_cols = detect_datetime_cols(df)
if dt_cols:
    print(f"[INFO] Datetime cols detected: {dt_cols}")
    df, added_time = add_time_features(df, dt_cols)
    print(f"[INFO] Added time features: {len(added_time)}")
else:
    print("[INFO] No datetime columns detected.")

# 3) 类别列与低基数整数列识别
object_cols = [c for c in df.columns if df[c].dtype == "object" and c != target]
low_card_int_cols = get_low_card_int_as_cat(df, target)
special_cat_cols = cast_special_categoricals(df)  # zip_code 等转为字符串类别
# 更新 object_cols（包含转字符串后的 special）
object_cols = sorted(list(set(object_cols + special_cat_cols)))
# 低基数整数列也作为类别（若已在object_cols则跳过）
candidate_cat_cols = sorted(list(set(object_cols + low_card_int_cols)))
print(f"[INFO] Cat candidates: {candidate_cat_cols}")

# 4) 缺失指示变量
added_miss_ind = add_missing_indicators(df)
added_miss_ind_test = add_missing_indicators(df_test)
print(f"[INFO] Added missing indicators: {len(added_miss_ind)}")

# 5) 比率特征
added_ratio = build_ratio_features(df)
added_ratio_test = build_ratio_features(df_test)
print(f"[INFO] Added ratio features: {added_ratio}")

# 6) 类别缺失填充
for c in candidate_cat_cols:
    if c in df.columns:
        df[c] = df[c].astype("string").fillna("Unknown")
        df_test[c] = df_test[c].astype("string").fillna("Unknown")





[INFO] Data shape: (53480, 19)
[INFO] Target column: label
[INFO] No datetime columns detected.
[INFO] Cat candidates: ['balance_accounts', 'career', 'installment', 'interest_rate', 'level', 'loan', 'residence', 'syndicated', 'term', 'title', 'total_accounts', 'zip_code']
[INFO] Added missing indicators: 1
[INFO] Added ratio features: ['payment_to_balance']
[INFO] Class weights: [0.6134153055606533, 2.7042880258899675] (pos_rate=0.184892)


In [5]:
df.to_csv('train/newtrain.csv',index=False)
df_test.to_csv('testaa/newtest.csv',index=False)

In [ ]:
# 7) 构建训练矩阵
features = [c for c in df.columns if c != target]
X = df[features].copy()
y = df[target].astype(int).values

# CatBoost 需要类别列索引
cat_features_idx = [X.columns.get_loc(c) for c in candidate_cat_cols if c in X.columns]

# 8) 类别不平衡权重
pos_rate = float(np.mean(y))
w_pos = 0.5 / pos_rate
w_neg = 0.5 / (1.0 - pos_rate)
class_weights = [w_neg, w_pos]
print(f"[INFO] Class weights: {class_weights} (pos_rate={pos_rate:.6f})")
# 9) 交叉验证训练
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
oof_pred = np.zeros(len(X), dtype=float)
fold_metrics = []

params = dict(
    loss_function="Logloss",
    eval_metric="AUC",
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=3.0,
    iterations=3000,
    random_seed=RANDOM_SEED,
    od_type="Iter",
    od_wait=200,
    class_weights=class_weights,
    verbose=200,
    allow_const_label=True,
    task_type="GPU" if USE_GPU else "CPU"
)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    X_tr, y_tr = X.iloc[tr_idx], y[tr_idx]
    X_va, y_va = X.iloc[va_idx], y[va_idx]

    train_pool = Pool(X_tr, y_tr, cat_features=cat_features_idx)
    valid_pool = Pool(X_va, y_va, cat_features=cat_features_idx)

    model = CatBoostClassifier(**params)
    model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

    pred_va = model.predict_proba(valid_pool)[:, 1]
    oof_pred[va_idx] = pred_va

    auc = roc_auc_score(y_va, pred_va)
    ks  = ks_score(y_va, pred_va)
    # 0.5 阈值下的分类指标
    yhat = (pred_va >= 0.5).astype(int)
    f1 = f1_score(y_va, yhat)
    p  = precision_score(y_va, yhat)
    r  = recall_score(y_va, yhat)

    fold_metrics.append({"fold": fold, "AUC": auc, "KS": ks, "F1@0.5": f1, "Prec@0.5": p, "Rec@0.5": r})
    print(f"[FOLD {fold}] AUC={auc:.6f} | KS={ks:.6f} | F1={f1:.6f} | P={p:.6f} | R={r:.6f}")

    # 保存本折特征重要性
    fi = pd.DataFrame({"feature": X.columns, "importance": model.get_feature_importance(train_pool)})
    fi.sort_values("importance", ascending=False).to_csv(f"feature_importance_fold{fold}.csv", index=False)

    del train_pool, valid_pool, model
    gc.collect()

# 10) 汇总指标与OOF
metrics_df = pd.DataFrame(fold_metrics)
print("\n[CV SUMMARY]")
print(metrics_df)
print(metrics_df.mean(numeric_only=True))

pd.DataFrame({"oof_pred": oof_pred, "label": y}).to_csv("oof_preds.csv", index=False)
metrics_df.to_csv("cv_metrics.csv", index=False)

# 11) 全量训练最终模型（以均值最佳迭代数做近似）
# （如需更严谨，可在CV中记录 best_iteration 再取均值+10%）
final_model = CatBoostClassifier(**params)
full_pool = Pool(X, y, cat_features=cat_features_idx)
final_model.fit(full_pool)
final_model.save_model("catboost_model.cbm")

# 导出整体特征重要性
fi_all = pd.DataFrame({"feature": X.columns, "importance": final_model.get_feature_importance(full_pool)})
fi_all.sort_values("importance", ascending=False).to_csv("feature_importance.csv", index=False)

# 保存使用到的列
with open("features_used.json", "w", encoding="utf-8") as f:
    json.dump({
        "features": features,
        "cat_features": [X.columns[i] for i in cat_features_idx]
    }, f, ensure_ascii=False, indent=2)

print("\n[OUTPUT]")
print(" - oof_preds.csv")
print(" - cv_metrics.csv")
print(" - feature_importance.csv & feature_importance_fold*.csv")
print(" - catboost_model.cbm")
print(" - features_used.json")


NameError: name 'df' is not defined